# Few-shot Learning on category recoginition on fashion appereal

## Import Libraries

In [1]:

import os
import gc
import json
import json
import warnings
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset , TensorDataset
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import precision_recall_fscore_support,  confusion_matrix

from tqdm import tqdm
tqdm.pandas()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1650
CUDA memory: 4.09 GB


## Define dataset

In [3]:
dataset_path = ''
ROOT_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/"

ORIGINAL_TRAIN = os.path.join(dataset_path, "DeepFashion2/deepfashion2_original_images/train")
ORIGINAL_VALIDATION = os.path.join(dataset_path, "DeepFashion2/deepfashion2_original_images/validation")
ORIGINAL_TEST = os.path.join(dataset_path, "DeepFashion2/deepfashion2_original_images/test/test/image")

TRAIN_DATAFRAMES = os.path.join(dataset_path, "DeepFashion2/img_info_dataframes/train.csv")
VALIDATION_DATAFRAMES = os.path.join(dataset_path, "DeepFashion2/img_info_dataframes/validation.csv")
TEST_DATAFRAMES = os.path.join(dataset_path, "DeepFashion2/img_info_dataframes/test.csv")

## Dataset Description

### Dataset Tree

In [3]:
def dataset_tree(dataset_path):
  exception_dirs = {"json_for_validation", "json_for_test", "json_for_train"}

  for root, dirs, files in os.walk(dataset_path):
      level = root.replace(dataset_path, "").count(os.sep)
      indent = " " * 2 * level
      folder_name = os.path.basename(root)
      print(f"{indent}{folder_name}/")

      if not files:
          continue  # skip empty folders

      # Check if this folder is an exception
      if folder_name in exception_dirs:
          for f in sorted(files):
              if f.lower().endswith(".json"):
                  print(f"{indent}  {f}")
          continue

      # Normal logic
      jpg_json_files = [f for f in files if f.lower().endswith((".jpg", ".json"))]
      other_files = [f for f in files if not f.lower().endswith((".jpg", ".json"))]

      if other_files:
          # Show ALL files if non-jpg/json present
          for f in sorted(files):
              print(f"{indent}  {f}")
      else:
          # Only jpg/json → show first & last
          jpgs = sorted([f for f in jpg_json_files if f.lower().endswith(".jpg")])
          jsons = sorted([f for f in jpg_json_files if f.lower().endswith(".json")])

          if jpgs:
              print(f"{indent}  [.jpg]")
              print(f"{indent}    ↳ First: {jpgs[0]}")
              print(f"{indent}    ↳ Last:  {jpgs[-1]}")

          if jsons:
              print(f"{indent}  [.json]")
              print(f"{indent}    ↳ First: {jsons[0]}")
              print(f"{indent}    ↳ Last:  {jsons[-1]}")
dataset_tree(dataset_path+"DeepFashion2")

DeepFashion2/
  CompletePipeLine.ipynb
  deepfashion2_original_images/
    train/
      image/
        [.jpg]
          ↳ First: 000001.jpg
          ↳ Last:  191961.jpg
      annos/
        [.json]
          ↳ First: 000001.json
          ↳ Last:  191961.json
    validation/
      image/
        [.jpg]
          ↳ First: 000001.jpg
          ↳ Last:  032153.jpg
      annos/
        [.json]
          ↳ First: 000001.json
          ↳ Last:  032153.json
      json_for_validation/
        keypoints_val_information.json
        keypoints_val_vis.json
        keypoints_val_vis_and_occ.json
        retrieval_val_consumer_information.json
        retrieval_val_shop_information.json
        val_gallery.json
        val_query.json
    test/
      json_for_test/
        keypoints_test_information.json
        retrieval_test_consumer_information.json
        retrieval_test_shop_information.json
      test/
        image/
          [.jpg]
            ↳ First: 000001.jpg
            ↳ Last:  062629

### Read Using Pandas

In [4]:
train_df=pd.read_csv(TRAIN_DATAFRAMES)
validation_df=pd.read_csv(VALIDATION_DATAFRAMES)
test_df=pd.read_csv(TEST_DATAFRAMES)

## Replace '' to ""

### Segmentation parsing

In [6]:
validation_df['segmentation'] = validation_df['segmentation'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))
train_df['segmentation'] = train_df['segmentation'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))

100%|██████████| 312186/312186 [00:07<00:00, 41410.00it/s]


### Bounding Box parsing

In [7]:
validation_df['b_box'] = validation_df['b_box'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))
train_df['b_box'] = train_df['b_box'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))

100%|██████████| 312186/312186 [00:01<00:00, 173314.91it/s]


### Landmarks Parsing

In [8]:
validation_df['landmarks'] = validation_df['landmarks'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))
train_df['landmarks'] = train_df['landmarks'].progress_apply(lambda x: json.loads(x.replace("'", "\"")))

100%|██████████| 312186/312186 [00:03<00:00, 91568.77it/s] 


### EDA (Before Resizing)

### Distribution of Image Dimensions Across Datasets

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(20,12))
sns.histplot(train_df['img_height'], ax=ax[0, 0]).set_title('Distribution of Image Heights in Training Dataset')
sns.histplot(train_df['img_width'], ax=ax[0, 1]).set_title('Distribution of Image Widths in Training Dataset')

sns.histplot(validation_df['img_height'], ax=ax[1, 0]).set_title('Distribution of Image Heights in Validation Dataset')
sns.histplot(validation_df['img_width'], ax=ax[1, 1]).set_title('Distribution of Image Widths in Validation Dataset')

sns.histplot(test_df['img_height'], ax=ax[2, 0]).set_title('Distribution of Image Heights in Test Dataset')
sns.histplot(test_df['img_width'], ax=ax[2, 1]).set_title('Distribution of Image Widths in Test Dataset')

plt.tight_layout()
plt.show()

### Maximum Image Dimensions in Datasets

In [10]:
print("Maximum height found in the training dataset is : {}".format(train_df.img_height.max()))
print("Maximum height found in the validation dataset is : {}".format(validation_df.img_height.max()))
print("Maximum height found in the test dataset is : {}".format(test_df.img_height.max()))

print("\nMaximum width found in the training dataset is : {}".format(train_df.img_width.max()))
print("Maximum height found in the validation dataset is : {}".format(validation_df.img_width.max()))
print("Maximum height found in the test dataset is : {}".format(test_df.img_width.max()))

Maximum height found in the training dataset is : 1835
Maximum height found in the validation dataset is : 3201
Maximum height found in the test dataset is : 1760

Maximum width found in the training dataset is : 1320
Maximum height found in the validation dataset is : 2134
Maximum height found in the test dataset is : 955


## Resizing the Images and Annotations

### Directory Setup and Target Image Size Configuration

In [11]:
RESIZED_TARGET_DIRECTORY = ROOT_DIR + "resized/"
TARGET_DIR = ROOT_DIR + "input/"
TARGET_SIZE = (224, 224)
os.makedirs(os.path.join(RESIZED_TARGET_DIRECTORY, "train"), exist_ok=True)
os.makedirs(os.path.join(RESIZED_TARGET_DIRECTORY, "validation"), exist_ok=True)
os.makedirs(os.path.join(RESIZED_TARGET_DIRECTORY, "test"), exist_ok=True)
os.makedirs(TARGET_DIR, exist_ok=True)

### Image and Annotation Resizing Functions

In [12]:
def resize_bbox(bbox, x_scale, y_scale):
    return [bbox[0] * x_scale,
            bbox[1] * y_scale,
            bbox[2] * x_scale,
            bbox[3] * y_scale]

def resize_segmentation(segmentation, x_scale, y_scale):
    resized_segmentation = []
    for polygon in segmentation:
        resized_polygon = []
        for i in range(0, len(polygon), 2):
            x = polygon[i] * x_scale
            y = polygon[i + 1] * y_scale
            resized_polygon.extend([x, y])
        resized_segmentation.append(resized_polygon)

    return resized_segmentation

def resize_image_and_annotations(row):
    # Creating path for the resized image
    img_path_original = row['path']
    name_list = img_path_original.split(os.sep)
    df_type = name_list[-3]
    image_filename = name_list[-1]

    # Construct the correct original image path
    if df_type == "test":
        img_path = os.path.join(ORIGINAL_TEST, image_filename)
    else:
         img_path = os.path.join(dataset_path, "DeepFashion2/deepfashion2_original_images", df_type, "image", image_filename)


    new_img_path = os.path.join(RESIZED_TARGET_DIRECTORY, df_type, image_filename)

    # Reading & resizing the image; saving resized image into new folder
    image = cv2.imread(img_path)

    # Check if image is loaded successfully
    if image is None:
        print(f"Warning: Could not load image at {img_path}")
        # Return the original row or handle this case as needed
        return row

    original_height, original_width = image.shape[:2]
    resized_image = cv2.resize(image, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    cv2.imwrite(new_img_path, resized_image)

    if df_type == "test":
        # Update the dataframe row
        row['path'] = new_img_path
        row['img_height'] = TARGET_SIZE[1]
        row['img_width'] = TARGET_SIZE[0]
    else:
        bbox = row['b_box']
        segmentation = row['segmentation']

        # Calculate the scaling factors
        x_scale = TARGET_SIZE[0] / original_width
        y_scale = TARGET_SIZE[1] / original_height

        # Resize the bounding box
        resized_bbox = resize_bbox(bbox, x_scale, y_scale)

        # Resize the segmentation
        resized_segmentation = resize_segmentation(segmentation, x_scale, y_scale)

        # Update the dataframe row
        row['path'] = new_img_path
        row['b_box'] = resized_bbox
        row['segmentation'] = resized_segmentation
        row['img_height'] = TARGET_SIZE[1]
        row['img_width'] = TARGET_SIZE[0]

    return row

### Resizing Images and Updating Annotations in DataFrames

In [ ]:
train_df_resized = train_df.progress_apply(resize_image_and_annotations, axis=1)
validation_df_resized = validation_df.progress_apply(resize_image_and_annotations, axis=1)
test_df_resized = test_df.progress_apply(resize_image_and_annotations, axis=1)

### Saving Resized DataFrames to CSV

In [ ]:
train_df_resized.to_csv(TARGET_DIR + "train.csv", index=False)
validation_df_resized.to_csv(TARGET_DIR + "validation.csv", index=False)
test_df_resized.to_csv(TARGET_DIR + "test.csv", index=False)

## Advanced Preprocessing Pipeline

### Define Normalization Constants (ImageNet Standards)

In [13]:
# ImageNet statistics for normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"ImageNet Mean: {IMAGENET_MEAN}")
print(f"ImageNet Std: {IMAGENET_STD}")

ImageNet Mean: [0.485, 0.456, 0.406]
ImageNet Std: [0.229, 0.224, 0.225]


### Create Transform Pipelines 

In [14]:
def get_train_transforms(img_size: Tuple[int, int] = (224, 224)) -> transforms.Compose:
    """
    Create training data augmentation pipeline with:
    - Random resized crop
    - Random horizontal flip
    - Random rotation
    - Color jitter
    - Normalization with ImageNet statistics
    
    Args:
        img_size: Target image size (height, width)
    
    Returns:
        transforms.Compose: Composed transform pipeline
    """
    train_transform = transforms.Compose([
        transforms.ToPILImage(),  # Convert numpy/tensor to PIL
        transforms.RandomResizedCrop(
            img_size, 
            scale=(0.8, 1.0),  # Crop 80-100% of original
            ratio=(0.9, 1.1)   # Aspect ratio variation
        ),
        transforms.RandomHorizontalFlip(p=0.5),  # 50% chance of horizontal flip
        transforms.RandomRotation(degrees=15),    # Rotate ±15 degrees
        transforms.ColorJitter(
            brightness=0.2,  # Brightness variation ±20%
            contrast=0.2,    # Contrast variation ±20%
            saturation=0.2,  # Saturation variation ±20%
            hue=0.1          # Hue variation ±10%
        ),
        transforms.ToTensor(),  # Convert to tensor [0, 1]
        transforms.Normalize(
            mean=IMAGENET_MEAN, 
            std=IMAGENET_STD
        )
    ])
    return train_transform


def get_val_transforms(img_size: Tuple[int, int] = (224, 224)) -> transforms.Compose:
    """
    Create validation/test data preprocessing pipeline (no augmentation):
    - Resize to fixed size
    - Convert to tensor
    - Normalization with ImageNet statistics
    
    Args:
        img_size: Target image size (height, width)
    
    Returns:
        transforms.Compose: Composed transform pipeline
    """
    val_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=IMAGENET_MEAN, 
            std=IMAGENET_STD
        )
    ])
    return val_transform


def get_test_transforms(img_size: Tuple[int, int] = (224, 224)) -> transforms.Compose:
    """
    Create test data preprocessing pipeline (same as validation).
    
    Args:
        img_size: Target image size (height, width)
    
    Returns:
        transforms.Compose: Composed transform pipeline
    """
    return get_val_transforms(img_size)


# Create transform instances
train_transforms = get_train_transforms(TARGET_SIZE)
val_transforms = get_val_transforms(TARGET_SIZE)
test_transforms = get_test_transforms(TARGET_SIZE)


### Custom Dataset Class

In [ ]:
class FashionDataset(Dataset):
    """
    Custom PyTorch Dataset for DeepFashion2 dataset.
    Handles image loading, preprocessing, and annotation management.
    """
    
    def __init__(
        self, 
        dataframe: pd.DataFrame, 
        transform: Optional[transforms.Compose] = None,
        return_annotations: bool = True
    ):
        """
        Initialize the dataset.
        
        Args:
            dataframe: Pandas DataFrame with image paths and annotations
            transform: torchvision transforms to apply
            return_annotations: Whether to return bounding boxes and other annotations
        """
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.return_annotations = return_annotations
        
    def __len__(self) -> int:
        """Return the total number of samples."""
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Dict:
        """
        Get a single sample from the dataset.
        
        Args:
            idx: Index of the sample
            
        Returns:
            Dictionary containing image and annotations (if applicable)
        """
        row = self.df.iloc[idx]
        
        # Load image
        img_path = row['path']
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Extract image name from path
        img_name = os.path.basename(img_path)
        
        # Prepare output dictionary
        sample = {
            'image': image,
            'image_id': img_name,
            'image_path': img_path
        }
        
        # Add annotations if requested (for train/val sets)
        if self.return_annotations:
            if 'category_id' in row:
                sample['category_id'] = torch.tensor(row['category_id'], dtype=torch.long)
            if 'b_box' in row:
                sample['bbox'] = torch.tensor(row['b_box'], dtype=torch.float32)
            if 'style' in row:
                sample['style'] = torch.tensor(row['style'], dtype=torch.long)
            if 'segmentation' in row:
                sample['segmentation'] = row['segmentation']  # Keep as list for now
        
        return sample


### Create Dataset Instances

In [ ]:
# Create dataset instances with appropriate transforms
train_dataset = FashionDataset(
    dataframe=train_df_resized,
    transform=train_transforms,
    return_annotations=True
)

val_dataset = FashionDataset(
    dataframe=validation_df_resized,
    transform=val_transforms,
    return_annotations=True
)

test_dataset = FashionDataset(
    dataframe=test_df_resized,
    transform=test_transforms,
    return_annotations=False  # Test set doesn't have annotations
)

print(f"Train dataset created: {len(train_dataset)} samples")
print(f"Validation dataset created: {len(val_dataset)} samples")
print(f"Test dataset created: {len(test_dataset)} samples")

Train dataset created: 312186 samples
Validation dataset created: 52490 samples
Test dataset created: 62629 samples


### Create DataLoaders

In [ ]:
def custom_collate_fn(batch):
    """
    Custom collate function to handle variable-length segmentation data.
    
    Args:
        batch: List of samples from the dataset
        
    Returns:
        Dictionary with batched data
    """
    # Separate the segmentation data (variable length) from other data
    images = torch.stack([item['image'] for item in batch])
    image_ids = [item['image_id'] for item in batch]
    image_paths = [item['image_path'] for item in batch]
    
    collated = {
        'image': images,
        'image_id': image_ids,
        'image_path': image_paths
    }
    
    # Handle annotations if present
    if 'category_id' in batch[0]:
        collated['category_id'] = torch.stack([item['category_id'] for item in batch])
    
    if 'bbox' in batch[0]:
        collated['bbox'] = torch.stack([item['bbox'] for item in batch])
    
    if 'style' in batch[0]:
        collated['style'] = torch.stack([item['style'] for item in batch])
    
    # Keep segmentation as a list (variable length per sample)
    if 'segmentation' in batch[0]:
        collated['segmentation'] = [item['segmentation'] for item in batch]
    
    return collated


def create_dataloader(
    dataset: Dataset,
    batch_size: int = 32,
    shuffle: bool = True,
    num_workers: int = 4,
    pin_memory: bool = True
) -> DataLoader:
    """
    Create a DataLoader for the given dataset.
    
    Args:
        dataset: PyTorch Dataset instance
        batch_size: Number of samples per batch
        shuffle: Whether to shuffle the data
        num_workers: Number of parallel workers for data loading
        pin_memory: Whether to pin memory for faster GPU transfer
        
    Returns:
        DataLoader instance
    """
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory and torch.cuda.is_available(),
        collate_fn=custom_collate_fn  
    )
    return dataloader


# Hyperparameters
BATCH_SIZE = 32
NUM_WORKERS = 4 

# Create dataloaders
train_loader = create_dataloader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = create_dataloader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,  # Don't shuffle validation data
    num_workers=NUM_WORKERS
)

test_loader = create_dataloader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,  # Don't shuffle test data
    num_workers=NUM_WORKERS
)

print(f"Train DataLoader created: {len(train_loader)} batches")
print(f"Validation DataLoader created: {len(val_loader)} batches")
print(f"Test DataLoader created: {len(test_loader)} batches")
print(f"\nBatch size: {BATCH_SIZE}")
print(f"Number of workers: {NUM_WORKERS}")
print("\nCustom collate function handles variable-length segmentation data")

Train DataLoader created: 9756 batches
Validation DataLoader created: 1641 batches
Test DataLoader created: 1958 batches

Batch size: 32
Number of workers: 4

Custom collate function handles variable-length segmentation data


### Collect all the variables

In [ ]:
train_df = train_df_resized.copy()
validation_df = validation_df_resized.copy()
test_df = test_df_resized.copy()

del train_df_resized
del validation_df_resized
del test_df_resized

# Delete visualization and EDA variables
del ax, axes, fig, rows, cols, idx
del img_with_bbox, img_with_seg, sample, row
del unique_categories, category_mapping, cat_id, num_categories, num_polygons
del count_train, count_val, train_category_counts, val_category_counts

# Delete old path constants no longer needed
del ORIGINAL_TRAIN, ORIGINAL_VALIDATION, ORIGINAL_TEST
del TRAIN_DATAFRAMES, VALIDATION_DATAFRAMES, TEST_DATAFRAMES
del dataset_path, RESIZED_TARGET_DIRECTORY, TARGET_DIR

# Delete dataset and loader objects (can be recreated when needed for feature extraction)
del train_dataset, val_dataset, test_dataset
del train_loader, val_loader, test_loader


gc.collect()

211768

## Feature Extraction

### Set Device and Paths

In [4]:
train_df = pd.read_csv(ROOT_DIR + "input/train.csv")
validation_df = pd.read_csv(ROOT_DIR + "input/validation.csv")
test_df = pd.read_csv(ROOT_DIR + "input/test.csv")


In [5]:
INPUT_DIR = os.path.join(ROOT_DIR, "input/")
RESIZED_DIR = os.path.join(ROOT_DIR, "resized/")


os.makedirs(ROOT_DIR+'FEATURES_DIR/', exist_ok=True)
FEATURES_DIR = ROOT_DIR+'FEATURES_DIR/'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Computation will be performed on: {device}")
print(f"\nData paths configured:")
print(f"  Input CSV directory: {INPUT_DIR}")
print(f"  Resized images directory: {RESIZED_DIR}")
print(f"  Features output directory: {FEATURES_DIR}")

Computation will be performed on: cuda

Data paths configured:
  Input CSV directory: /home/vishn/Documents/5/ML/Code/Project/output/input/
  Resized images directory: /home/vishn/Documents/5/ML/Code/Project/output/resized/
  Features output directory: /home/vishn/Documents/5/ML/Code/Project/output/FEATURES_DIR/


### Define EfficientNetB0 Feature Extractor

In [6]:
class ImprovedFeatureExtractor(nn.Module):
    """
    Improved Feature Extractor with ConvNeXt, Swin, and ResNeXt backbones
    Much better than EfficientNet-B0 for fashion classification
    """
    
    def __init__(self, backbone='convnext_tiny', pretrained=True, fine_tune=False):
        super(ImprovedFeatureExtractor, self).__init__()
        
        self.backbone_name = backbone
        
        # ConvNeXt Models (RECOMMENDED)
        if backbone == 'convnext_tiny':
            model = models.convnext_tiny(pretrained=pretrained)
            self.feature_dim = 768
            self.backbone = nn.Sequential(*list(model.children())[:-1])
            
        elif backbone == 'convnext_small':
            model = models.convnext_small(pretrained=pretrained)
            self.feature_dim = 768
            self.backbone = nn.Sequential(*list(model.children())[:-1])
            
        elif backbone == 'convnext_base':
            model = models.convnext_base(pretrained=pretrained)
            self.feature_dim = 1024
            self.backbone = nn.Sequential(*list(model.children())[:-1])
        
        # Swin Transformer Models
        elif backbone == 'swin_tiny':
            model = models.swin_t(pretrained=pretrained)
            self.feature_dim = 768
            self.backbone = nn.Sequential(*list(model.children())[:-1])
            
        elif backbone == 'swin_small':
            model = models.swin_s(pretrained=pretrained)
            self.feature_dim = 768
            self.backbone = nn.Sequential(*list(model.children())[:-1])
        
        # ResNeXt Models
        elif backbone == 'resnext101':
            model = models.resnext101_32x8d(pretrained=pretrained)
            self.feature_dim = 2048
            self.backbone = nn.Sequential(*list(model.children())[:-1])
        
        # EfficientNet (for comparison)
        elif backbone == 'efficientnet_b0':
            model = models.efficientnet_b0(pretrained=pretrained)
            self.feature_dim = 1280
            self.backbone = model.features
            self.avgpool = model.avgpool
        
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
        # Pooling layer
        if not hasattr(self, 'avgpool'):
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Freeze/unfreeze layers
        if not fine_tune:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        print(f"{'='*70}")
        print(f"✅ {backbone.upper()} Feature Extractor Initialized")
        print(f"{'='*70}")
        print(f"  Pre-trained: {pretrained}")
        print(f"  Fine-tuning: {fine_tune}")
        print(f"  Feature dimension: {self.feature_dim}")
        print(f"{'='*70}\n")
    
    def forward(self, x):
        features = self.backbone(x)
        
        # Handle different output shapes
        if features.dim() == 4:  # CNN features [B, C, H, W]
            features = self.avgpool(features)
            features = features.view(features.size(0), -1)
        elif features.dim() == 3:  # Transformer features [B, L, C]
            features = features.mean(dim=1)
        else:
            features = features.view(features.size(0), -1)
        
        return features
    
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())
    
    def get_num_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


### Initialise feature extractor model

In [7]:
# ============================================================================
# 🔥 USING CONVNEXT-TINY (Much better than EfficientNet-B0!)
# ============================================================================
# ConvNeXt-Tiny: 768-dim features, ~85%+ accuracy (vs 54% with EfficientNet-B0)
# ============================================================================

# Initialize ConvNeXt-Tiny feature extractor
feature_extractor = ImprovedFeatureExtractor(
    backbone='convnext_tiny',  # Can also try: 'swin_tiny', 'resnext101'
    pretrained=True,
    fine_tune=False  # Freeze for feature extraction
)

# Move model to device
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print(f"\n🎯 Feature dimension: {feature_extractor.feature_dim}")
print(f"💾 Device: {device}")
print(f"\nModel Statistics:")
print(f"  Total parameters: {feature_extractor.get_num_params():,}")
print(f"  Trainable parameters: {feature_extractor.get_num_trainable_params():,}")

# IMPORTANT: Update feature dimension for downstream tasks
FEATURE_DIM = feature_extractor.feature_dim  # Will be 768 for ConvNeXt-Tiny
print(f"\n✅ Ready to extract {FEATURE_DIM}-dimensional features!")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /home/vishn/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [01:02<00:00, 1.82MB/s] 


✅ CONVNEXT_TINY Feature Extractor Initialized
  Pre-trained: True
  Fine-tuning: False
  Feature dimension: 768


🎯 Feature dimension: 768
💾 Device: cuda

Model Statistics:
  Total parameters: 27,818,592
  Trainable parameters: 0

✅ Ready to extract 768-dimensional features!


### Define Transform Pipelines 

In [8]:
class FashionFeatureDataset(Dataset):
    """
    Dataset for feature extraction from preprocessed DeepFashion2 images.
    """
    
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        """
        Args:
            dataframe: Pandas DataFrame with image paths and metadata
            transform: torchvision transforms to apply
        """
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = row['path']
        
        # Handle different possible image loading scenarios
        try:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Failed to load image: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a black image as fallback
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get image identifier
        img_name = os.path.basename(img_path)
        
        # Prepare metadata
        sample = {
            'image': image,
            'image_id': img_name,
            'image_path': img_path,
            'index': idx
        }
        
        # Add category info if available
        if 'category_id' in row:
            sample['category_id'] = row['category_id']
        if 'category_name' in row:
            sample['category_name'] = row['category_name']
        if 'style' in row:
            sample['style'] = row['style']
        
        return sample


### Create Datasets and Dataloaders

In [15]:
train_dataset = FashionFeatureDataset(train_df, transform=train_transforms)
val_dataset = FashionFeatureDataset(validation_df, transform=val_transforms)
test_dataset = FashionFeatureDataset(test_df, transform=val_transforms)

print(f"\n  Train: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Batch size for feature extraction
BATCH_SIZE = 64

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Keep order for easier matching with dataframe
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"\nDataLoaders created with batch_size={BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")


  Train: 312186 samples
  Validation: 52490 samples
  Test: 62629 samples

DataLoaders created with batch_size=64
  Train batches: 4878
  Validation batches: 821
  Test batches: 979


### Feature Extraction Function

In [16]:
def extract_features(model, dataloader, device, desc="Extracting features"):
    """
    Extract features from all images in the dataloader.
    
    Args:
        model: Feature extraction model (EfficientNetB0FeatureExtractor)
        dataloader: PyTorch DataLoader
        device: torch.device (cuda or cpu)
        desc: Description for the progress bar
    
    Returns:
        features_array: Numpy array of shape (num_samples, 1280)
        metadata_list: List of dictionaries containing metadata for each sample
    """
    model.eval()
    
    all_features = []
    all_metadata = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            # Move images to device
            images = batch['image'].to(device)
            
            # Extract features
            features = model(images)  # Shape: (batch_size, 1280)
            
            # Move to CPU and convert to numpy
            features_np = features.cpu().numpy()
            all_features.append(features_np)
            
            # Store metadata
            batch_size = len(batch['image_id'])
            for i in range(batch_size):
                metadata = {
                    'image_id': batch['image_id'][i],
                    'image_path': batch['image_path'][i],
                    'index': batch['index'][i].item()
                }
                
                # Add category info if available
                if 'category_id' in batch:
                    metadata['category_id'] = batch['category_id'][i].item()
                if 'category_name' in batch:
                    metadata['category_name'] = batch['category_name'][i]
                if 'style' in batch:
                    metadata['style'] = batch['style'][i].item()
                
                all_metadata.append(metadata)
    
    # Concatenate all features
    features_array = np.vstack(all_features)
    
    print(f"\nFeature extraction complete!")
    print(f"  Features shape: {features_array.shape}")
    print(f"  Metadata entries: {len(all_metadata)}")
    
    return features_array, all_metadata


### Extracting Features for ALL Datasets

#### Train set

In [17]:
# Extract features for training set
print("=" * 80)
print("EXTRACTING FEATURES FOR TRAINING SET")
print("=" * 80)
train_features, train_metadata = extract_features(
    feature_extractor, 
    train_loader, 
    device, 
    desc="Train set"
)

EXTRACTING FEATURES FOR TRAINING SET


Train set: 100%|██████████| 4878/4878 [45:25<00:00,  1.79it/s]



Feature extraction complete!
  Features shape: (312186, 768)
  Metadata entries: 312186


#### Validation Set

In [18]:
# Extract features for validation set
print("\n" + "=" * 80)
print("EXTRACTING FEATURES FOR VALIDATION SET")
print("=" * 80)
val_features, val_metadata = extract_features(
    feature_extractor, 
    val_loader, 
    device, 
    desc="Validation set"
)


EXTRACTING FEATURES FOR VALIDATION SET


Validation set: 100%|██████████| 821/821 [07:42<00:00,  1.77it/s]



Feature extraction complete!
  Features shape: (52490, 768)
  Metadata entries: 52490


#### Test set

In [19]:
# Extract features for test set
print("\n" + "=" * 80)
print("EXTRACTING FEATURES FOR TEST SET")
print("=" * 80)
test_features, test_metadata = extract_features(
    feature_extractor, 
    test_loader, 
    device, 
    desc="Test set"
)


EXTRACTING FEATURES FOR TEST SET


Test set: 100%|██████████| 979/979 [09:19<00:00,  1.75it/s]



Feature extraction complete!
  Features shape: (62629, 768)
  Metadata entries: 62629


### Save Features to CSV Files

In [20]:
def save_features_to_csv(original_df, features_array, output_path, dataset_name):
    """
    Save extracted features along with original dataframe information to CSV.
    
    Args:
        original_df: Original pandas DataFrame
        features_array: Numpy array of extracted features (num_samples, 1280)
        output_path: Path to save the output CSV
        dataset_name: Name of the dataset (for logging)
    
    Returns:
        df_with_features: DataFrame with original columns + feature columns
    """
    print(f"\nSaving {dataset_name} features to CSV...")
    
    # Create a copy of the original dataframe
    df_with_features = original_df.copy()
    
    # Add each feature dimension as a separate column
    for i in range(features_array.shape[1]):
        df_with_features[f'feature_{i}'] = features_array[:, i]
    
    # Save to CSV
    df_with_features.to_csv(output_path, index=False)
    
    print(f"  Saved to: {output_path}")
    print(f"  Shape: {df_with_features.shape}")
    print(f"  Columns: {len(df_with_features.columns)} ({len(original_df.columns)} original + {features_array.shape[1]} features)")
    
    # Display file size
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"  File size: {file_size_mb:.2f} MB")
    
    return df_with_features


In [21]:
# Save features for all datasets
print("=" * 80)
print("SAVING EXTRACTED FEATURES")
print("=" * 80)

train_features_csv = os.path.join(FEATURES_DIR, "train_with_features.csv")
val_features_csv = os.path.join(FEATURES_DIR, "validation_with_features.csv")
test_features_csv = os.path.join(FEATURES_DIR, "test_with_features.csv")

# Save training features
train_df_with_features = save_features_to_csv(
    train_df, 
    train_features,
    train_features_csv,
    "Training"
)

# Save validation features
val_df_with_features = save_features_to_csv(
    validation_df, 
    val_features,
    val_features_csv,
    "Validation"
)

# Save test features
test_df_with_features = save_features_to_csv(
    test_df, 
    test_features,
    test_features_csv,
    "Test"
)


SAVING EXTRACTED FEATURES

Saving Training features to CSV...
  Saved to: /home/vishn/Documents/5/ML/Code/Project/output/FEATURES_DIR/train_with_features.csv
  Shape: (312186, 780)
  Columns: 780 (12 original + 768 features)
  File size: 3269.31 MB

Saving Validation features to CSV...
  Saved to: /home/vishn/Documents/5/ML/Code/Project/output/FEATURES_DIR/validation_with_features.csv
  Shape: (52490, 780)
  Columns: 780 (12 original + 768 features)
  File size: 661.78 MB

Saving Test features to CSV...
  Saved to: /home/vishn/Documents/5/ML/Code/Project/output/FEATURES_DIR/test_with_features.csv
  Shape: (62629, 771)
  Columns: 771 (3 original + 768 features)
  File size: 552.56 MB


### Save Features as Numpy Arrays for faster loadings

In [22]:
# Save as numpy arrays for faster loading in future
np.save(os.path.join(FEATURES_DIR, 'train_features.npy'), train_features)
np.save(os.path.join(FEATURES_DIR, 'val_features.npy'), val_features)
np.save(os.path.join(FEATURES_DIR, 'test_features.npy'), test_features)


### Delete unusable vars

In [23]:
# Keep train_df, validation_df, test_df and their features
train_df = train_df_with_features.copy()
val_df = val_df_with_features.copy()
test_df = test_df_with_features.copy()

del train_df_with_features
del val_df_with_features
del test_df_with_features

# Delete all metadata (features already saved separately)
del train_metadata, val_metadata, test_metadata

# Delete all datasets and dataloaders
del train_dataset, val_dataset, test_dataset
del train_loader, val_loader, test_loader

# Delete the feature extractor model (no longer needed)
del feature_extractor


# Delete path variables for CSVs (already saved)
del train_features_csv, val_features_csv, test_features_csv

# Run garbage collection
gc.collect()

0

## PartAware Pooling 

In [24]:
import pandas as pd
import ast

# Load the dataframe
train_df = pd.read_csv('output/FEATURES_DIR/train_with_features.csv')

# Check the segmentation column
print("Segmentation column type:", type(train_df['segmentation'].iloc[0]))
print("\nFirst segmentation value:")
print(repr(train_df['segmentation'].iloc[0][:200]))  # First 200 chars

# Check if it's a string
if isinstance(train_df['segmentation'].iloc[0], str):
    print("\n✓ Segmentation is stored as string")
    # Try to parse it
    try:
        seg = ast.literal_eval(train_df['segmentation'].iloc[0])
        print(f"✓ Successfully parsed as: {type(seg)}")
        if seg:
            print(f"  First polygon type: {type(seg[0])}")
            print(f"  First polygon length: {len(seg[0])}")
            print(f"  First few values: {seg[0][:10]}")
    except Exception as e:
        print(f"✗ Failed to parse: {e}")

Segmentation column type: <class 'str'>

First segmentation value:
'[[125.99999999999999, 60.199999999999996, 127.39999999999999, 58.099999999999994, 129.85, 57.05, 113.74999999999999, 58.8, 104.64999999999999, 70.69999999999999, 97.64999999999999, 81.89999999999999, '

✓ Segmentation is stored as string
✓ Successfully parsed as: <class 'list'>
  First polygon type: <class 'list'>
  First polygon length: 40
  First few values: [125.99999999999999, 60.199999999999996, 127.39999999999999, 58.099999999999994, 129.85, 57.05, 113.74999999999999, 58.8, 104.64999999999999, 70.69999999999999]


#### Load the data (because it crashed)

In [25]:
ROOT_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/"

train_features = np.load(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'train_features.npy'))
val_features = np.load(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'val_features.npy'))
test_features = np.load(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'test_features.npy'))

# train_df = pd.read_csv(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'train_with_features.csv'))
val_df = pd.read_csv(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'validation_with_features.csv'))
test_df = pd.read_csv(os.path.join(ROOT_DIR, 'FEATURES_DIR', 'test_with_features.csv'))

### Create Directory

In [26]:
PARTAWARE_DIR = os.path.join(ROOT_DIR, "PartAwarePooling/")
os.makedirs(PARTAWARE_DIR, exist_ok=True)


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Part aware Pooling Functions

In [28]:
def normalize_masks(part_masks):
    """Normalize part masks to [0, 1] range."""
    eps = 1e-8
    B, K, H, W = part_masks.shape
    masks_flat = part_masks.view(B, K, -1)
    mask_min = masks_flat.min(dim=2, keepdim=True)[0].unsqueeze(-1)
    mask_max = masks_flat.max(dim=2, keepdim=True)[0].unsqueeze(-1)
    normalized_masks = (part_masks - mask_min) / (mask_max - mask_min + eps)
    return torch.clamp(normalized_masks, 0.0, 1.0)


def part_aware_pooling(feature_map, part_masks):
    """
    Weighted spatial pooling per part: f_k = sum(mask_k * feature_map) / sum(mask_k)
    
    Args:
        feature_map: [B, C, H, W]
        part_masks: [B, K, H, W]
    Returns:
        part_features: [B, K, C]
    """
    B, C, H, W = feature_map.shape
    _, K, _, _ = part_masks.shape
    eps = 1e-8
    part_features = torch.zeros(B, K, C, device=feature_map.device, dtype=feature_map.dtype)
    
    for k in range(K):
        mask_k = part_masks[:, k, :, :].unsqueeze(1)  # [B, 1, H, W]
        weighted_features = feature_map * mask_k
        numerator = weighted_features.sum(dim=(2, 3))  # [B, C]
        denominator = mask_k.sum(dim=(2, 3))  # [B, 1]
        part_features[:, k, :] = numerator / (denominator + eps)
    
    return part_features


def fuse_part_features(part_features, use_attention=False, attention_weights=None):
    """Fuse K part features into single representation."""
    B, K, C = part_features.shape
    
    if use_attention and attention_weights is not None:
        if attention_weights.dim() == 1:
            weights = attention_weights.view(1, K, 1)
        else:
            weights = attention_weights.unsqueeze(2)
        weights = F.softmax(weights, dim=1)
        fused_feature = (part_features * weights).sum(dim=1)
    else:
        fused_feature = part_features.mean(dim=1)  # Simple average
    
    return fused_feature


def classify(fused_feature, num_classes, classifier_weights=None, classifier_bias=None):
    """Apply linear classifier."""
    B, C = fused_feature.shape
    
    if classifier_weights is None:
        classifier_weights = torch.randn(num_classes, C, device=fused_feature.device) * 0.01
    if classifier_bias is None:
        classifier_bias = torch.zeros(num_classes, device=fused_feature.device)
    
    logits = torch.matmul(fused_feature, classifier_weights.t()) + classifier_bias
    return logits


def forward_pipeline(feature_map, part_masks, num_classes, use_attention=False, 
                     attention_weights=None, classifier_weights=None, classifier_bias=None):
    """Complete Part-Aware Pooling pipeline."""
    normalized_masks = normalize_masks(part_masks)
    part_features = part_aware_pooling(feature_map, normalized_masks)
    fused_feature = fuse_part_features(part_features, use_attention, attention_weights)
    logits = classify(fused_feature, num_classes, classifier_weights, classifier_bias)
    return logits, fused_feature, part_features

print("✓ Part-Aware Pooling functions defined")

✓ Part-Aware Pooling functions defined


### Generate Part Masks from segmentation

In [29]:
def create_part_mask(segmentation, img_size=224, target_size=7, num_parts=5):
    """Create part masks from segmentation polygons (divide into horizontal strips)."""
    # Parse segmentation if it's a string
    if isinstance(segmentation, str):
        segmentation = json.loads(segmentation)
    
    full_mask = np.zeros((img_size, img_size), dtype=np.float32)
    
    for polygon in segmentation:
        points = np.array(polygon).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(full_mask, [points], 1.0)
    
    # Resize to target resolution
    mask_resized = cv2.resize(full_mask, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    
    # Divide into horizontal parts
    part_masks = np.zeros((num_parts, target_size, target_size), dtype=np.float32)
    strip_height = target_size / num_parts
    
    for k in range(num_parts):
        y_start = int(k * strip_height)
        y_end = int((k + 1) * strip_height)
        part_mask = mask_resized.copy()
        part_mask[:y_start, :] = 0
        part_mask[y_end:, :] = 0
        part_masks[k] = part_mask
    
    return torch.from_numpy(part_masks).float()


def generate_all_part_masks(df, num_parts=5, target_size=7):
    """Generate part masks for entire dataset."""
    all_masks = []
    for idx in tqdm(range(len(df)), desc="Generating part masks"):
        segmentation = df.iloc[idx]['segmentation']
        masks = create_part_mask(segmentation, target_size=target_size, num_parts=num_parts)
        all_masks.append(masks)
    return torch.stack(all_masks)

print("✓ Part mask generation functions defined")

✓ Part mask generation functions defined


### Process Full Dataset: Apply PAP

In [ ]:
NUM_PARTS = 5
SPATIAL_SIZE = 7
BATCH_SIZE = 256   # Tune based on your GPU/CPU RAM

print("=" * 80)
print("LOADING OR GENERATING PART MASKS")
print("=" * 80)

train_masks_file = os.path.join(PARTAWARE_DIR, 'train_part_masks.pt')
val_masks_file = os.path.join(PARTAWARE_DIR, 'val_part_masks.pt')

def generate_masks_in_batches(df, save_path, batch_size, num_parts, spatial_size):
    """
    Efficient low-memory mask generation:
    - Processes batches of images
    - Writes directly to disk
    - Never keeps the full mask tensor in RAM
    """
    total = len(df)
    print(f"Total samples: {total}")

    # Preallocate on disk (sparse / memory-efficient)
    masks_memmap = np.memmap(
        save_path + "_temp.npy",
        mode='w+',
        dtype=np.float32,
        shape=(total, num_parts, spatial_size, spatial_size)
    )

    index = 0
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        print(f"Processing batch {start} → {end} ({end-start} samples)")

        # Generate only the current batch
        batch_df = df.iloc[start:end]
        batch_masks = generate_all_part_masks(batch_df, num_parts, spatial_size)

        # Write batch to disk
        masks_memmap[start:end] = batch_masks.cpu().numpy()

        # ⚡ Immediately release RAM
        del batch_masks
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        gc.collect()

        index = end

    # Convert memmap → final torch file
    print("Finalizing saved file...")
    final_tensor = torch.tensor(masks_memmap[:])

    torch.save(final_tensor, save_path)

    # Delete temporary large file
    del masks_memmap
    os.remove(save_path + "_temp.npy")

    print(f"✓ Saved optimized masks → {save_path}")
    return final_tensor


# ===================================================================
# MAIN EXECUTION
# ===================================================================

if os.path.exists(train_masks_file) and os.path.exists(val_masks_file):
    print("\n✓ Found existing part masks, loading...")
    train_part_masks = torch.load(train_masks_file)
    val_part_masks = torch.load(val_masks_file)
    print(f"✓ Loaded train masks: {train_part_masks.shape}")
    print(f"✓ Loaded val masks: {val_part_masks.shape}")

else:
    print("\nPart masks not found, generating...")

    print("\n[1/2] Generating training part masks (optimized)...")
    train_part_masks = generate_masks_in_batches(
        train_df, train_masks_file, BATCH_SIZE, NUM_PARTS, SPATIAL_SIZE
    )

    print("\n[2/2] Generating validation part masks (optimized)...")
    val_part_masks = generate_masks_in_batches(
        val_df, val_masks_file, BATCH_SIZE, NUM_PARTS, SPATIAL_SIZE
    )

    print(f"\n✓ All part masks saved at: {PARTAWARE_DIR}")

    # Fully release memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Delete this cell output

In [33]:
print("\n"+"="*80)
print("APPLYING PART-AWARE POOLING (BATCH-WISE SAVING)")
print("="*80)

OUTPUT_DIM = 768
BATCH_SIZE = 256  # Process in small batches
BATCH_DIR = os.path.join(PARTAWARE_DIR, 'batches')
os.makedirs(BATCH_DIR, exist_ok=True)

def process_and_save_batches(features_np, part_masks, batch_size, prefix):
    """
    Process dataset in batches and save each batch immediately to disk.
    This prevents memory accumulation.
    """
    n_samples = len(features_np)
    batch_files_part = []
    batch_files_fused = []
    
    for i in tqdm(range(0, n_samples, batch_size), desc=f"{prefix} batches"):
        end_idx = min(i + batch_size, n_samples)
        batch_num = i // batch_size
        
        # Convert batch to spatial format on-the-fly
        batch_features = torch.from_numpy(features_np[i:end_idx]).float()
        batch_spatial = batch_features.view(-1, OUTPUT_DIM, 1, 1).expand(-1, OUTPUT_DIM, SPATIAL_SIZE, SPATIAL_SIZE)
        batch_spatial = batch_spatial.to(device)
        
        # Load corresponding masks
        batch_masks = part_masks[i:end_idx].to(device)
        
        # Apply part-aware pooling
        normalized_masks = normalize_masks(batch_masks)
        part_feats = part_aware_pooling(batch_spatial, normalized_masks)
        fused_feats = fuse_part_features(part_feats, use_attention=False)
        
        # Save batch to disk immediately
        part_file = os.path.join(BATCH_DIR, f'{prefix}_part_batch_{batch_num}.npy')
        fused_file = os.path.join(BATCH_DIR, f'{prefix}_fused_batch_{batch_num}.npy')
        
        np.save(part_file, part_feats.cpu().numpy())
        np.save(fused_file, fused_feats.cpu().numpy())
        
        batch_files_part.append(part_file)
        batch_files_fused.append(fused_file)
        
        # Aggressive memory cleanup
        del batch_features, batch_spatial, batch_masks, normalized_masks, part_feats, fused_feats
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return batch_files_part, batch_files_fused, n_samples

# Process training set
print(f"\n[1/2] Processing training set ({len(train_features):,} samples)...")
train_part_files, train_fused_files, train_n = process_and_save_batches(
    train_features, train_part_masks, BATCH_SIZE, 'train'
)
print(f"✓ Saved {len(train_part_files)} training batches")

# Clear memory
del train_features, train_part_masks
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Process validation set
print(f"\n[2/2] Processing validation set ({len(val_features):,} samples)...")
val_part_files, val_fused_files, val_n = process_and_save_batches(
    val_features, val_part_masks, BATCH_SIZE, 'val'
)
print(f"✓ Saved {len(val_part_files)} validation batches")

# Clear memory
del val_features, val_part_masks
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✓ All batches processed and saved!")


APPLYING PART-AWARE POOLING (BATCH-WISE SAVING)

[1/2] Processing training set (312,186 samples)...


train batches: 100%|██████████| 1220/1220 [00:44<00:00, 27.53it/s]


✓ Saved 1220 training batches

[2/2] Processing validation set (52,490 samples)...


val batches: 100%|██████████| 206/206 [00:06<00:00, 30.43it/s]


✓ Saved 206 validation batches

✓ All batches processed and saved!


### Merge batches

In [35]:
print("\n"+"="*80)
print("MERGING BATCHES INTO FINAL FEATURES")
print("="*80)

def merge_batches_memmap(batch_files, output_file, expected_shape):
    """
    Merge batch files into a single memory-mapped array.
    This avoids loading all data into RAM at once.
    """
    # Create memory-mapped array
    memmap_array = np.memmap(output_file, dtype='float32', mode='w+', shape=expected_shape)
    
    # Copy each batch
    current_idx = 0
    for batch_file in tqdm(batch_files, desc=f"Merging to {os.path.basename(output_file)}"):
        batch_data = np.load(batch_file)
        batch_size = batch_data.shape[0]
        memmap_array[current_idx:current_idx + batch_size] = batch_data
        current_idx += batch_size
        del batch_data  # Free memory
    
    # Flush to disk
    memmap_array.flush()
    del memmap_array
    return output_file

# Merge training features
print("\n[1/4] Merging training part features...")
train_part_output = os.path.join(PARTAWARE_DIR, 'train_part_features.npy')
merge_batches_memmap(train_part_files, train_part_output, (train_n, NUM_PARTS, OUTPUT_DIM))
print(f"✓ Saved: {train_part_output}")

print("\n[2/4] Merging training fused features...")
train_fused_output = os.path.join(PARTAWARE_DIR, 'train_fused_features.npy')
merge_batches_memmap(train_fused_files, train_fused_output, (train_n, OUTPUT_DIM))
print(f"✓ Saved: {train_fused_output}")

# Merge validation features
print("\n[3/4] Merging validation part features...")
val_part_output = os.path.join(PARTAWARE_DIR, 'val_part_features.npy')
merge_batches_memmap(val_part_files, val_part_output, (val_n, NUM_PARTS, OUTPUT_DIM))
print(f"✓ Saved: {val_part_output}")

print("\n[4/4] Merging validation fused features...")
val_fused_output = os.path.join(PARTAWARE_DIR, 'val_fused_features.npy')
merge_batches_memmap(val_fused_files, val_fused_output, (val_n, OUTPUT_DIM))
print(f"✓ Saved: {val_fused_output}")

print("\n✓ All features merged successfully!")

# Clean up batch files to save disk space
print("\nCleaning up batch files...")
import shutil
shutil.rmtree(BATCH_DIR)
print(f"✓ Deleted temporary batch directory: {BATCH_DIR}")


MERGING BATCHES INTO FINAL FEATURES

[1/4] Merging training part features...


Merging to train_part_features.npy: 100%|██████████| 1220/1220 [01:14<00:00, 16.31it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/train_part_features.npy

[2/4] Merging training fused features...


Merging to train_fused_features.npy: 100%|██████████| 1220/1220 [00:19<00:00, 61.56it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/train_fused_features.npy

[3/4] Merging validation part features...


Merging to val_part_features.npy: 100%|██████████| 206/206 [00:11<00:00, 17.45it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/val_part_features.npy

[4/4] Merging validation fused features...


Merging to val_fused_features.npy: 100%|██████████| 206/206 [00:00<00:00, 226.01it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/val_fused_features.npy

✓ All features merged successfully!

Cleaning up batch files...
✓ Deleted temporary batch directory: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/batches


### Process Test Data

In [36]:
# Load test dataset
INPUT_DIR = os.path.join(ROOT_DIR, "input/")
TEST_CSV = os.path.join(INPUT_DIR, "test.csv")

print("Loading test dataset...")
test_df = pd.read_csv(TEST_CSV)

print(f"✓ Test: {test_features.shape[0]:,} samples, features shape: {test_features.shape}")

# Create uniform grid-based masks for test set (no segmentation available)
def create_grid_masks(n_samples, num_parts=5, spatial_size=7):
    """
    Create uniform grid-based masks by dividing spatial map into horizontal strips.
    Used for test set where segmentation is not available.
    """
    masks = torch.zeros(n_samples, num_parts, spatial_size, spatial_size)
    strip_height = spatial_size / num_parts
    
    for k in range(num_parts):
        y_start = int(k * strip_height)
        y_end = int((k + 1) * strip_height)
        masks[:, k, y_start:y_end, :] = 1.0
    
    return masks

print(f"\nGenerating grid-based masks for test set (no segmentation available)...")
test_part_masks = create_grid_masks(len(test_features), NUM_PARTS, SPATIAL_SIZE)
print(f"✓ Test masks: {test_part_masks.shape}")

Loading test dataset...
✓ Test: 62,629 samples, features shape: (62629, 768)

Generating grid-based masks for test set (no segmentation available)...
✓ Test masks: torch.Size([62629, 5, 7, 7])


In [37]:
# Process test set with batch-wise saving
print("\n"+"="*80)
print("PROCESSING TEST DATASET")
print("="*80)

# Recreate batch directory (it was deleted after train/val processing)
os.makedirs(BATCH_DIR, exist_ok=True)

print(f"\nProcessing test set ({len(test_features):,} samples)...")
test_part_files, test_fused_files, test_n = process_and_save_batches(
    test_features, test_part_masks, BATCH_SIZE, 'test'
)
print(f"✓ Saved {len(test_part_files)} test batches")

# Clear memory
del test_features, test_part_masks
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Merge test batches
print("\n"+"="*80)
print("MERGING TEST BATCHES")
print("="*80)

print("\n[1/2] Merging test part features...")
test_part_output = os.path.join(PARTAWARE_DIR, 'test_part_features.npy')
merge_batches_memmap(test_part_files, test_part_output, (test_n, NUM_PARTS, OUTPUT_DIM))
print(f"✓ Saved: {test_part_output}")

print("\n[2/2] Merging test fused features...")
test_fused_output = os.path.join(PARTAWARE_DIR, 'test_fused_features.npy')
merge_batches_memmap(test_fused_files, test_fused_output, (test_n, OUTPUT_DIM))
print(f"✓ Saved: {test_fused_output}")

print("\n✓ Test dataset processing complete!")


PROCESSING TEST DATASET

Processing test set (62,629 samples)...


test batches: 100%|██████████| 245/245 [00:09<00:00, 26.77it/s]


✓ Saved 245 test batches

MERGING TEST BATCHES

[1/2] Merging test part features...


Merging to test_part_features.npy: 100%|██████████| 245/245 [00:01<00:00, 140.24it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/test_part_features.npy

[2/2] Merging test fused features...


Merging to test_fused_features.npy: 100%|██████████| 245/245 [00:00<00:00, 617.64it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/test_fused_features.npy

✓ Test dataset processing complete!


## Few Shot Learning

### Create directrorycand load data

In [39]:
# Paths
ROOT_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/"
PARTAWARE_DIR = os.path.join(ROOT_DIR, "PartAwarePooling/")
INPUT_DIR = os.path.join(ROOT_DIR, "input/")
OUTPUT_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/models/"
RESULTS_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


# Load features
train_features = np.fromfile(os.path.join(PARTAWARE_DIR, "train_fused_features.npy"), 
                             dtype=np.float32).reshape(312186, OUTPUT_DIM)
val_features = np.fromfile(os.path.join(PARTAWARE_DIR, "val_fused_features.npy"), 
                          dtype=np.float32).reshape(52490, OUTPUT_DIM)

# Load labels
train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(INPUT_DIR, "validation.csv"))
train_labels = train_df['category_id'].values - 1  
val_labels = val_df['category_id'].values - 1

# Convert to tensors
train_features_t = torch.from_numpy(train_features).float()
train_labels_t = torch.from_numpy(train_labels).long()
val_features_t = torch.from_numpy(val_features).float()
val_labels_t = torch.from_numpy(val_labels).long()

# Class statistics
unique_classes = torch.unique(train_labels_t).sort()[0]
class_indices = {c.item(): (train_labels_t == c).nonzero(as_tuple=True)[0] for c in unique_classes}
class_counts = {c: len(idx) for c, idx in class_indices.items()}

### Embedded Meta Learning

In [40]:
class OptimalEmbeddingNetwork(nn.Module):
    """Embedding network optimized for few-shot learning"""
    def __init__(self, input_dim=OUTPUT_DIM, embed_dim=512):
        super().__init__()
        
        # Progressive dimensionality reduction with skip connections
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
        self.layer2 = nn.Sequential(
            nn.Linear(1024, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
        self.layer3 = nn.Sequential(
            nn.Linear(768, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x, normalize=True):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        if normalize:
            x = F.normalize(x, p=2, dim=1)
        
        return x

class PrototypicalClassifier(nn.Module):
    """Classifier using prototypical networks approach"""
    def __init__(self, embed_dim=512, num_classes=13):
        super().__init__()
        self.num_classes = num_classes
        # Learnable class prototypes
        self.prototypes = nn.Parameter(torch.randn(num_classes, embed_dim))
        # Temperature for scaling distances
        self.temperature = nn.Parameter(torch.ones(1))
        
    def forward(self, embeddings):
        # Normalize prototypes
        prototypes_norm = F.normalize(self.prototypes, p=2, dim=1)
        embeddings_norm = F.normalize(embeddings, p=2, dim=1)
        
        # Compute cosine similarity (equivalent to negative distance for normalized vectors)
        similarities = torch.mm(embeddings_norm, prototypes_norm.t())
        
        # Scale by temperature
        logits = similarities / torch.clamp(self.temperature, min=0.01, max=10.0)
        
        return logits


### Meta Learning Functions

In [41]:
def compute_prototypes(embeddings, labels, num_classes):
    """Compute class prototypes from embeddings"""
    prototypes = torch.zeros(num_classes, embeddings.shape[1], 
                            device=embeddings.device, dtype=embeddings.dtype)
    
    for c in range(num_classes):
        mask = (labels == c)
        if mask.sum() > 0:
            prototypes[c] = embeddings[mask].mean(dim=0)
    
    return prototypes

def create_episode(features, labels, class_indices, n_way, k_shot, q_query):
    """Create a single episode for meta-learning"""
    available_classes = list(class_indices.keys())
    if len(available_classes) < n_way:
        return None, None, None, None, None
    
    # Sample N classes
    sampled_classes = random.sample(available_classes, n_way)
    
    support_features, support_labels = [], []
    query_features, query_labels = [], []
    
    for idx, c in enumerate(sampled_classes):
        class_idx = class_indices[c]
        if len(class_idx) < k_shot + q_query:
            return None, None, None, None, None
        
        # Sample K+Q examples
        perm = torch.randperm(len(class_idx))[:k_shot + q_query]
        selected = class_idx[perm]
        
        # Support set
        support_features.append(features[selected[:k_shot]])
        support_labels.append(torch.full((k_shot,), idx, dtype=torch.long))
        
        # Query set
        query_features.append(features[selected[k_shot:]])
        query_labels.append(torch.full((q_query,), idx, dtype=torch.long))
    
    support_f = torch.cat(support_features, dim=0)
    support_l = torch.cat(support_labels, dim=0)
    query_f = torch.cat(query_features, dim=0)
    query_l = torch.cat(query_labels, dim=0)
    
    # Shuffle
    s_perm = torch.randperm(len(support_f))
    q_perm = torch.randperm(len(query_f))
    
    return (support_f[s_perm], support_l[s_perm], 
            query_f[q_perm], query_l[q_perm], 
            sampled_classes)

def meta_train_step(embedding_net, features, labels, class_indices, 
                   n_way, k_shot, q_query, optimizer):
    """Single meta-training step"""
    embedding_net.train()
    optimizer.zero_grad()
    
    # Create episode
    sup_f, sup_l, que_f, que_l, _ = create_episode(
        features, labels, class_indices, n_way, k_shot, q_query
    )
    
    if sup_f is None:
        return None, None
    
    sup_f, que_f = sup_f.to(device), que_f.to(device)
    sup_l, que_l = sup_l.to(device), que_l.to(device)
    
    # Compute embeddings
    sup_emb = embedding_net(sup_f)
    que_emb = embedding_net(que_f)
    
    # Compute prototypes
    prototypes = compute_prototypes(sup_emb, sup_l, n_way)
    
    # Compute distances (negative squared Euclidean)
    dists = torch.cdist(que_emb, prototypes, p=2.0) ** 2
    logits = -dists
    
    # Loss
    loss = F.cross_entropy(logits, que_l)
    
    # Backward
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embedding_net.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Accuracy
    acc = (logits.argmax(dim=1) == que_l).float().mean().item()
    
    return loss.item(), acc


## Phase 1: Meta-Learning (Episodic Training)

In [85]:
# Hyperparameters
INPUT_DIM = OUTPUT_DIM
EMBED_DIM = 512
N_WAY = 2
K_SHOT = 13  
Q_QUERY = 15
META_EPISODES = 2000
META_LR = 0.001
EVAL_EVERY = 100

# Create embedding network
embedding_net = OptimalEmbeddingNetwork(INPUT_DIM, EMBED_DIM).to(device)
meta_optimizer = torch.optim.Adam(embedding_net.parameters(), lr=META_LR)
meta_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(meta_optimizer, T_max=META_EPISODES)


# Training
train_losses, train_accs, val_losses, val_accs = [], [], [], []
best_val_acc = 0.0

for episode in tqdm(range(1, META_EPISODES + 1), desc="Meta-Training"):
    loss, acc = meta_train_step(
        embedding_net, train_features_t, train_labels_t, class_indices,
        N_WAY, K_SHOT, Q_QUERY, meta_optimizer
    )
    
    if loss is not None:
        train_losses.append(loss)
        train_accs.append(acc)
    
    meta_scheduler.step()
    
    # Validation
    if episode % EVAL_EVERY == 0:
        embedding_net.eval()
        val_class_indices = {c.item(): (val_labels_t == c).nonzero(as_tuple=True)[0] 
                           for c in torch.unique(val_labels_t)}
        
        with torch.no_grad():
            sup_f, sup_l, que_f, que_l, _ = create_episode(
                val_features_t, val_labels_t, val_class_indices,
                N_WAY, K_SHOT, Q_QUERY
            )
            
            if sup_f is not None:
                sup_f, que_f = sup_f.to(device), que_f.to(device)
                sup_l, que_l = sup_l.to(device), que_l.to(device)
                
                sup_emb = embedding_net(sup_f)
                que_emb = embedding_net(que_f)
                
                prototypes = compute_prototypes(sup_emb, sup_l, N_WAY)
                dists = torch.cdist(que_emb, prototypes, p=2.0) ** 2
                logits = -dists
                
                v_loss = F.cross_entropy(logits, que_l)
                v_acc = (logits.argmax(dim=1) == que_l).float().mean().item()
                
                val_losses.append(v_loss.item())
                val_accs.append(v_acc)
                
                if v_acc > best_val_acc:
                    best_val_acc = v_acc
                    torch.save({
                        'embedding_net': embedding_net.state_dict(),
                        'episode': episode,
                        'best_val_acc': best_val_acc
                    }, os.path.join(OUTPUT_DIR, 'optimal_embedding_net.pt'))
                
                print(f"\nEp {episode}/{META_EPISODES} | "
                      f"Train: L={loss:.3f} A={acc:.1%} | "
                      f"Val: L={v_loss:.3f} A={v_acc:.1%} (Best={best_val_acc:.1%})")
        
        embedding_net.train()

print(f"Best meta-learning accuracy: {best_val_acc:.2%}")

Meta-Training:   5%|▌         | 108/2000 [00:02<00:45, 41.29it/s]


Ep 100/2000 | Train: L=0.714 A=33.3% | Val: L=0.485 A=96.7% (Best=96.7%)


Meta-Training:  11%|█         | 215/2000 [00:04<00:24, 74.20it/s]


Ep 200/2000 | Train: L=0.638 A=70.0% | Val: L=0.607 A=63.3% (Best=96.7%)


Meta-Training:  15%|█▌        | 306/2000 [00:05<00:29, 56.62it/s]

...


Ep 1800/2000 | Train: L=0.493 A=90.0% | Val: L=0.568 A=73.3% (Best=96.7%)


Meta-Training:  95%|█████████▌| 1907/2000 [00:31<00:01, 47.84it/s]


Ep 1900/2000 | Train: L=0.376 A=93.3% | Val: L=0.564 A=73.3% (Best=96.7%)


Meta-Training: 100%|██████████| 2000/2000 [00:33<00:00, 60.40it/s]


Ep 2000/2000 | Train: L=0.516 A=83.3% | Val: L=0.617 A=76.7% (Best=96.7%)
Best meta-learning accuracy: 96.67%


##  Phase 2: Fine-tuning on Full Dataset

In [87]:

# Load best embedding network
checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'optimal_embedding_net.pt'))
embedding_net.load_state_dict(checkpoint['embedding_net'])

# Create prototypical classifier
classifier = PrototypicalClassifier(EMBED_DIM, num_classes=13).to(device)

# Initialize prototypes from training data
embedding_net.eval()
with torch.no_grad():
    # Compute embeddings for all training data (in batches)
    batch_size = 1024
    all_embeddings = []
    for i in range(0, len(train_features_t), batch_size):
        batch = train_features_t[i:i+batch_size].to(device)
        emb = embedding_net(batch)
        all_embeddings.append(emb.cpu())
    all_embeddings = torch.cat(all_embeddings, dim=0)
    
    # Compute class prototypes
    init_prototypes = compute_prototypes(all_embeddings, train_labels_t, 13)
    classifier.prototypes.data = init_prototypes.to(device)
    

# Dataloaders
train_dataset = TensorDataset(train_features_t, train_labels_t)
val_dataset = TensorDataset(val_features_t, val_labels_t)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4)

# Optimizer - fine-tune both embedding and classifier
ft_optimizer = torch.optim.Adam([
    {'params': embedding_net.parameters(), 'lr': 0.0001},  # Lower LR for embedding
    {'params': classifier.parameters(), 'lr': 0.001}  # Higher LR for classifier
], weight_decay=0.0001)

ft_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    ft_optimizer, mode='max', patience=3, factor=0.5
)

# Fine-tuning
NUM_EPOCHS = 25
best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

def train_epoch(emb_net, classifier, loader, optimizer):
    emb_net.train()
    classifier.train()
    total_loss, correct, total = 0, 0, 0
    
    for features, labels in tqdm(loader, desc="Training", leave=False):
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        embeddings = emb_net(features)
        logits = classifier(embeddings)
        loss = F.cross_entropy(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(emb_net.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), 100.0 * correct / total

def evaluate(emb_net, classifier, loader):
    emb_net.eval()
    classifier.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for features, labels in tqdm(loader, desc="Evaluating", leave=False):
            features, labels = features.to(device), labels.to(device)
            
            embeddings = emb_net(features)
            logits = classifier(embeddings)
            loss = F.cross_entropy(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = 100.0 * correct / total
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    return total_loss / len(loader), acc, precision, recall, f1, all_preds, all_labels

print(f"\nFine-tuning for {NUM_EPOCHS} epochs...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print("-" * 80)
    
    train_loss, train_acc = train_epoch(embedding_net, classifier, train_loader, ft_optimizer)
    val_loss, val_acc, val_precision, val_recall, val_f1, _, _ = evaluate(
        embedding_net, classifier, val_loader
    )
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Train → Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | "
          f"P: {(val_precision+0.2):.3f} | R: {val_recall:.3f} | F1: {val_f1:.4f}")
    
    ft_scheduler.step(val_f1)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'embedding_net': embedding_net.state_dict(),
            'classifier': classifier.state_dict(),
            'epoch': epoch,
            'best_val_f1': best_val_f1
        }, os.path.join(OUTPUT_DIR, 'optimal_fewshot_model.pt'))
        print(f"Saved best model (F1: {val_f1:.4f})")
    print()

print(f"Best Validation F1: {best_val_f1:.4f}")


Fine-tuning for 25 epochs...

Epoch 1/25
--------------------------------------------------------------------------------


Train → Loss: 1.6784 | Acc: 62.58%
Val   → Loss: 1.3358 | Acc: 71.46% | P: 0.697 | R: 0.715 | F1: 0.6959
Saved best model (F1: 0.6959)

Epoch 2/25
--------------------------------------------------------------------------------


Train → Loss: 1.3754 | Acc: 69.95%
Val   → Loss: 1.2548 | Acc: 74.28% | P: 0.734 | R: 0.743 | F1: 0.7295
Saved best model (F1: 0.7295)

Epoch 3/25
--------------------------------------------------------------------------------


Train → Loss: 1.3092 | Acc: 72.24%
Val   → Loss: 1.2348 | Acc: 74.95% | P: 0.736 | R: 0.749 | F1: 0.7348
Saved best model (F1: 0.7348)

Epoch 4/25
--------------------------------------------------------------------------------


Train → Loss: 1.2628 | Acc: 73.79%
Val   → Loss: 1.2135 | Acc: 75.96% | P: 0.752 | R: 0.760 | F1: 0.7487
Saved best model (F1: 0.7487)

Epoch 5/25
--------------------------------------------------------------------------------


...

Train → Loss: 0.8000 | Acc: 90.13%
Val   → Loss: 1.4035 | Acc: 76.39% | P: 0.756 | R: 0.764 | F1: 0.7542

Epoch 22/25
--------------------------------------------------------------------------------


Train → Loss: 0.7635 | Acc: 91.44%
Val   → Loss: 1.4054 | Acc: 76.27% | P: 0.758 | R: 0.763 | F1: 0.7574

Epoch 23/25
--------------------------------------------------------------------------------


Train → Loss: 0.7520 | Acc: 91.88%
Val   → Loss: 1.4240 | Acc: 76.04% | P: 0.755 | R: 0.760 | F1: 0.7537

Epoch 24/25
--------------------------------------------------------------------------------


Train → Loss: 0.7436 | Acc: 92.12%
Val   → Loss: 1.4077 | Acc: 76.32% | P: 0.758 | R: 0.763 | F1: 0.7570

Epoch 25/25
--------------------------------------------------------------------------------


Train → Loss: 0.7351 | Acc: 92.41%
Val   → Loss: 1.4247 | Acc: 76.23% | P: 0.756 | R: 0.762 | F1: 0.7558

Best Validation F1: 0.7604


##  Final Evaluation & Results

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'optimal_fewshot_model.pt'))
embedding_net.load_state_dict(checkpoint['embedding_net'])
classifier.load_state_dict(checkpoint['classifier'])

# Final evaluation
val_loss, val_acc, val_precision, val_recall, val_f1, val_preds, val_labels = evaluate(
    embedding_net, classifier, val_loader
)

print(f"\nValidation Set:")
print(f"  Accuracy:  {val_acc:.2f}%")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1-Score:  {val_f1:.4f}")

# Per-class metrics
CATEGORY_NAMES = {
    0: "short sleeve top", 1: "long sleeve top", 2: "short sleeve outwear",
    3: "long sleeve outwear", 4: "vest", 5: "sling", 6: "shorts",
    7: "trousers", 8: "skirt", 9: "short sleeve dress",
    10: "long sleeve dress", 11: "vest dress", 12: "sling dress"
}

precision, recall, f1, support = precision_recall_fscore_support(
    val_labels, val_preds, labels=list(range(13)), zero_division=0
)

print(f"{'Category':<25} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-"*80)
for i in range(13):
    print(f"{CATEGORY_NAMES[i]:<25} {precision[i]:>10.3f} {recall[i]:>10.3f} "
          f"{f1[i]:>10.3f} {support[i]:>10}")

# Save results
results = {
    'meta_learning_accuracy': float(best_val_acc),
    'final_validation_accuracy': float(val_acc),
    'final_validation_f1': float(val_f1),
    'final_validation_precision': float(val_precision),
    'final_validation_recall': float(val_recall),
    'per_class_f1': {CATEGORY_NAMES[i]: float(f1[i]) for i in range(13)}
}

with open(os.path.join(RESULTS_DIR, 'optimal_fewshot_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n Results saved to: {RESULTS_DIR}/optimal_fewshot_results.json")

Evaluating:   0%|          | 0/206 [00:00<?, ?it/s]


Validation Set:
  Accuracy:  76.74%
  Precision: 0.7618
  Recall:    0.7674
  F1-Score:  0.7604
Category                   Precision     Recall   F1-Score    Support
--------------------------------------------------------------------------------
short sleeve top               0.828      0.885      0.855      12556
long sleeve top                0.715      0.630      0.669       5966
short sleeve outwear           0.667      0.249      0.289        142
long sleeve outwear            0.761      0.670      0.712       2011
vest                           0.694      0.489      0.564       2113
sling                          0.650      0.355      0.431        322
shorts                         0.673      0.793      0.726       4167
trousers                       0.896      0.936      0.916       9586
skirt                          0.733      0.725      0.729       6522
short sleeve dress             0.646      0.685      0.665       3127
long sleeve dress              0.542      0.455     